In [35]:
# Import Data Manipulation Libraries
import pandas as pd
import numpy as np 
# Import FilterWarnings Libraries
import warnings
warnings.filterwarnings('ignore')
# Import Logging
import logging
logging.basicConfig(level = logging.INFO,
                    filename = 'model.log',
                    filemode = 'w',
                    format = '%(asctime)s - %(message)s - %(levelname)s',
                    force = True)
# Import Machine Learning Libraries
from sklearn.preprocessing import MinMaxScaler,RobustScaler,LabelEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [36]:
# Step1: Data Ingestion
def data_loader():
    df = pd.read_csv(r'C:\Devesh ITV\Machine Learning\CustomerChurnAnalysisModel\data\raw\churn_dataset.csv')
    return df

In [37]:
# Step2: Data Preprocessing 
def preprocessing(df):
    
    # step1: Dopping duplicates value 
    df.drop_duplicates()
    
    # step2: Segregate numerical columns and categorical columns
    numerical_data = df.select_dtypes(exclude = 'object').columns
    categorical_data = df.select_dtypes(include = 'object').columns
    
    # step3: Enconding Target Column
    df['Churn'] = df['Churn'].map({'Yes':1,'No':0})
    
    # Using Winsorization Technique 
    from scipy.stats.mstats import winsorize 
    
    for i in numerical_data:
        df[i] = winsorize(df[i],limits = [0.05,0.05])
    
    le = LabelEncoder()
    for i in categorical_data:
        df[i] = le.fit_transform(df[i])
        
    
    for i in numerical_data:
        df[i] = df[i].fillna(df[i].median())
        
    
    # step4: Split the dataset into X and y
    X = df.drop(columns = ['CustomerID', 'Churn','ServiceArea'])
    y = df['Churn']
    
    # step5: Using train and test split i.e. Seen Data and Unseen Data
    X_train,X_test,y_train,y_test = train_test_split(X,y,
                                                     train_size = 0.70,
                                                     random_state = 1)

    

    # step6: Use SMOTE Technique : Over Sampling Technique ---> KNN Based 
    smote = SMOTE()
    X_train,y_train = smote.fit_resample(X_train,y_train) # type: ignore
        
    # step7: Using Scaling Technique 
    sc = MinMaxScaler()
    X_train = sc.fit_transform(X_train)  # type: ignore # Seen Data
    X_test = sc.transform(X_test)   # Unseen Data
    
        # Using PCA
    from sklearn.decomposition import PCA
    pca = PCA(n_components= 0.95)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test) 
    
    return X_train,X_test,y_train,y_test
   

In [38]:
# Step3: Model Building
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

def model_building(X_train,X_test,y_train,y_test):
    model = RandomForestClassifier().fit(X_train,y_train)

    y_pred = model.predict(X_test)

    report_score = classification_report(y_test,y_pred)
    print(model.feature_importances_)
    return report_score
    

In [39]:
# Entry Point
def main():
    df = data_loader()
    print(df.shape)
    print(df.columns)
    X_train,X_test,y_train,y_test = preprocessing(df)
    print(X_train.shape,y_train.shape)
    report_score = model_building(X_train,X_test,y_train,y_test)
    print(report_score)

main()

(51047, 58)
Index(['CustomerID', 'Churn', 'MonthlyRevenue', 'MonthlyMinutes',
       'TotalRecurringCharge', 'DirectorAssistedCalls', 'OverageMinutes',
       'RoamingCalls', 'PercChangeMinutes', 'PercChangeRevenues',
       'DroppedCalls', 'BlockedCalls', 'UnansweredCalls', 'CustomerCareCalls',
       'ThreewayCalls', 'ReceivedCalls', 'OutboundCalls', 'InboundCalls',
       'PeakCallsInOut', 'OffPeakCallsInOut', 'DroppedBlockedCalls',
       'CallForwardingCalls', 'CallWaitingCalls', 'MonthsInService',
       'UniqueSubs', 'ActiveSubs', 'ServiceArea', 'Handsets', 'HandsetModels',
       'CurrentEquipmentDays', 'AgeHH1', 'AgeHH2', 'ChildrenInHH',
       'HandsetRefurbished', 'HandsetWebCapable', 'TruckOwner', 'RVOwner',
       'Homeownership', 'BuysViaMailOrder', 'RespondsToMailOffers',
       'OptOutMailings', 'NonUSTravel', 'OwnsComputer', 'HasCreditCard',
       'RetentionCalls', 'RetentionOffersAccepted', 'NewCellphoneUser',
       'NotNewCellphoneUser', 'ReferralsMadeBySubscriber'